# 07 — الأداء والـ Load (Performance)

بنقيس: **latency** للبحث (p50/p95/p99)، **concurrency** (throughput وأخطاء)، سرعة الـ **embedding**، سرعة الـ **ingestion**، وخطة تنفيذ الـ **pgvector** (`EXPLAIN ANALYZE`) — وبنقارن رقم الـ client مع اللي Prometheus سجّله.

> ⚠️ الاختبار ده in-process (FastAPI TestClient على CPU) فالأرقام **نسبية** للمقارنة قبل/بعد التعديلات مش أرقام production. للـ load الحقيقي استخدم `locust` أو `k6` ضد الـ containers.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("07_performance_and_load")

In [ ]:
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from sqlalchemy import text
from fastapi.testclient import TestClient
import numpy as np

api = get_api(); app = api.app
T = CONFIG["THRESHOLDS"]
FLOW = None
with rec.case("prepare data: ingest EN + AR documents", "setup") as c:
    FLOW = quick_flow(api, files=("orion_manual.txt", "blue_nile_plan_ar.txt"), chat=False)
    c.note({k: round(v, 1) for k, v in FLOW["timings"].items()})
need_flow = lambda: need(FLOW)

## 1) Latency للبحث (sequential)

In [ ]:
QUERIES = [q for q, *_ in QA[:6]]
lat, errors = [], 0
with rec.case("40 sequential searches", "latency") as c:
    need_flow(); u, pid = FLOW["user"], FLOW["project_id"]
    api.search(u, pid, QUERIES[0])                     # warm-up (model/cache)
    for i in range(40):
        t0 = time.perf_counter(); r = api.search(u, pid, QUERIES[i % len(QUERIES)]); lat.append((time.perf_counter() - t0) * 1000)
        errors += r.status_code != 200
    assert errors == 0, f"{errors} failed requests"
    c.note(f"p50={pct(lat, 50):.0f} p95={pct(lat, 95):.0f} p99={pct(lat, 99):.0f} max={max(lat):.0f} ms")
if lat:
    display(pd.Series({"min": min(lat), "p50": pct(lat, 50), "p95": pct(lat, 95), "p99": pct(lat, 99), "max": max(lat), "mean": np.mean(lat)}, name="ms").round(1).to_frame().T)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2)); ax[0].hist(lat, bins=15); ax[0].set_title("search latency (ms)"); ax[1].plot(lat); ax[1].set_title("latency over requests (look for drift/outliers)"); plt.tight_layout(); plt.show()
    rec.check(f"search p95 <= {T['search_p95_ms']} ms", pct(lat, 95) <= T["search_p95_ms"], f"p95={pct(lat, 95):.0f}ms", "latency")
    rec.check("no latency drift (last 10 mean < 2x first 10 mean)", np.mean(lat[-10:]) < 2 * np.mean(lat[:10]) + 5, f"first10={np.mean(lat[:10]):.0f}ms last10={np.mean(lat[-10:]):.0f}ms", "latency", level="warn")

## 2) Concurrency sweep
40 request لكل مستوى. بنراقب الـ throughput ومعدل الأخطاء والـ p95.

In [ ]:
def one_thread(args):
    n, uid, pid, token = args
    cl = TestClient(app, raise_server_exceptions=False); out = []
    for i in range(n):
        t0 = time.perf_counter()
        r = cl.request(api.routes["search"][0], fill(api.routes["search"][1], pid) if "{" in api.routes["search"][1] else api.routes["search"][1],
                       headers={"Authorization": f"Bearer {token}"}, json=PAYLOADS["search"](QUERIES[i % len(QUERIES)], pid) if api.routes["search"][0] != "GET" else None,
                       params=PAYLOADS["search"](QUERIES[i % len(QUERIES)], pid) if api.routes["search"][0] == "GET" else None)
        out.append(((time.perf_counter() - t0) * 1000, r.status_code))
    return out

SW = []
for lvl in (1, 4, 8, 16):
    with rec.case(f"concurrency={lvl}", "concurrency") as c:
        need_flow(); per = max(1, 40 // lvl); u = FLOW["user"]
        t0 = time.perf_counter()
        with ThreadPoolExecutor(lvl) as ex: res = [x for part in ex.map(one_thread, [(per, 0, FLOW["project_id"], u.token)] * lvl) for x in part]
        wall = time.perf_counter() - t0; ms = [m for m, _ in res]; err = sum(s != 200 for _, s in res)
        SW.append(dict(concurrency=lvl, requests=len(res), rps=round(len(res) / wall, 1), p50=round(pct(ms, 50)), p95=round(pct(ms, 95)), errors=err))
        c.note(f"{len(res) / wall:.1f} req/s, p95={pct(ms, 95):.0f}ms, errors={err}")
        assert err == 0, f"{err} errors under concurrency {lvl}"
if SW:
    S_DF = pd.DataFrame(SW); display(S_DF)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2)); S_DF.plot(x="concurrency", y="rps", marker="o", ax=ax[0], title="throughput (req/s)"); S_DF.plot(x="concurrency", y=["p50", "p95"], marker="o", ax=ax[1], title="latency (ms)"); plt.tight_layout(); plt.show()

## 3) سرعة الـ Embedding (الـ CPU)

In [ ]:
M = None
with rec.case("load embedding model", "embedding") as c:
    from sentence_transformers import SentenceTransformer
    M = SentenceTransformer(CONFIG["EMBEDDING_MODEL"]); c.note(f"max_seq_length={M.max_seq_length}")
EMB = []
passage = "passage: " + " ".join(["The reactor coolant circulates through two redundant pumps during production."] * 6)
for bs in (1, 8, 32):
    with rec.case(f"embedding throughput, batch={bs}", "embedding") as c:
        need(M); texts = [passage] * bs; M.encode(texts[:1])
        t0 = time.perf_counter(); reps = max(1, 32 // bs)
        for _ in range(reps): M.encode(texts, batch_size=bs)
        dt = time.perf_counter() - t0; tps = bs * reps / dt; EMB.append(dict(batch=bs, texts_per_s=round(tps, 1))); c.note(f"{tps:.1f} chunks/s")
if EMB: display(pd.DataFrame(EMB))

## 4) سرعة الـ Ingestion

In [ ]:
with rec.case("ingest a ~150 KB text file: time, chunks, throughput", "ingestion") as c:
    need_flow(); u, pid = FLOW["user"], FLOW["project_id"]
    big = "\n\n".join(f"Chapter {i}. " + " ".join(f"Sentence {j} about topic {i} and subject {j * 7 % 13}." for j in range(40)) for i in range(70)).encode()
    t0 = time.perf_counter(); r = api.upload(u, pid, "big_doc.txt", big, "text/plain"); up = time.perf_counter() - t0
    assert r.status_code in (200, 201, 202), f"HTTP {r.status_code}"
    fid = get_id(r); st, secs = api.wait_ready(u, fid, timeout=600); assert st in CONFIG["READY_STATUSES"], st
    with get_session() as s: n = chunk_stats(s, fid)["total"]
    c.note(f"{len(big) / 1024:.0f}KB -> {n} chunks in {secs + up:.1f}s = {len(big) / 1024 / (secs + up):.1f} KB/s, {n / (secs + up):.1f} chunks/s")
    assert n > 20, "unexpectedly few chunks"

## 5) pgvector: هل الـ ANN index بيتستخدم؟

In [ ]:
with rec.case("EXPLAIN ANALYZE of a nearest-neighbour query", "pgvector") as c:
    with get_session() as s:
        t, v = chunk_table(s); rows = s.execute(text(f'select count(*) from "{t}"')).scalar()
        q = "[" + ",".join(f"{x:.5f}" for x in (np.random.default_rng(0).standard_normal(CONFIG["EMBED_DIM"]) * 0.05)) + "]"
        plan = [r[0] for r in s.execute(text(f"EXPLAIN (ANALYZE, BUFFERS) select id from \"{t}\" order by \"{v}\" <=> '{q}'::vector limit 5")).all()]
    print("\n".join(plan))
    uses_idx = any("Index Scan" in l and ("hnsw" in l.lower() or "ivfflat" in l.lower() or "embedding" in l.lower()) for l in plan)
    exec_ms = next((float(re.search(r"Execution Time: ([\d.]+)", l).group(1)) for l in plan if "Execution Time" in l), None)
    c.note(f"rows={rows}", f"exec={exec_ms}ms", "ANN index used" if uses_idx else "sequential scan")
    if not uses_idx and rows > 5000: raise Warn(f"sequential scan on {rows} rows - create an HNSW index (CREATE INDEX ... USING hnsw ({v} vector_cosine_ops))")

with rec.case("memory: process RSS", "resources") as c:
    try:
        import psutil; rss = psutil.Process().memory_info().rss / 1e6
    except ImportError: raise Skip("pip install psutil")
    c.note(f"{rss:.0f} MB (includes notebook + embedding model)")

## 6) مقارنة رقم الـ client مع Prometheus
لو الـ histogram بتاع الـ server بيقول p95 مختلف جداً عن اللي قسناه، يبقى فيه مشكلة في الـ instrumentation (buckets غلط أو timer في مكان غلط).

In [ ]:
with rec.case("server-side p95 (from histogram buckets) is consistent with client-side p95", "prometheus") as c:
    need(lat or None)
    fams, flat = parse_metrics(scrape(api).text); D = detect_http(fams)
    if not D["hist"]: raise Skip("no HTTP histogram detected")
    sp = api.routes["search"][1]; tpl = sp
    bks = {}
    for (n, lab), v in flat.items():
        d = dict(lab)
        if n == D["hist"] + "_bucket" and d.get("method") == api.routes["search"][0] and (D["path_label"] is None or d.get(D["path_label"]) in (tpl, fill(tpl, FLOW["project_id"]))):
            le = float("inf") if d["le"] == "+Inf" else float(d["le"]); bks[le] = bks.get(le, 0) + v
    if not bks: raise Skip(f"no histogram series for {tpl}")
    q95 = hist_quantile(sorted(bks.items()), 0.95) * 1000
    c.note(f"server p95~{q95:.0f}ms (bucket-interpolated), client p95={pct(lat, 95):.0f}ms")
    assert q95 <= pct(lat, 95) * 3 + 100, "server histogram p95 is far above what clients measured -> check bucket layout / timer scope"
    if q95 < pct(lat, 95) * 0.2: raise Warn("server p95 is much lower than client p95: queueing/serialization outside the timed middleware, or coarse buckets")

In [ ]:
rec.summary(); rec.table(); rec.save()